# 01. FBref Data Ingestion

**Stage:** Ingestion  
**Inputs:** `soccerdata` FBref and ClubElo readers, EPL seasons from config  
**Outputs:** `data/raw/{season}/fixtures.parquet`, `data/raw/{season}/match_stats.parquet`, `data/raw/{season}/player_minutes.parquet`

This notebook uses the maintained `soccerdata` library instead of a custom website scraper. FBref supplies fixtures and match/player statistics; ClubElo is available as an additional rating input for later ML features.

In [1]:
# Load configuration and imports
from pathlib import Path
import sys

import pandas as pd

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "src" / "config" / "loader.py").exists():
    repo_root = repo_root.parent

sys.path.insert(0, str(repo_root))

from src.config.loader import load_config
from src.ingestion.soccerdata_client import SoccerDataClient, SoccerDataConfig

config = load_config()
season = config["data"]["seasons"][0]  # Start with first season
league_id = config["data"]["default_league_id"]

print(f"Season: {season}")
print(f"League ID: {league_id}")

[08/20/26 03:54:28] INFO     No custom team name replacements found. You can configure these in       ]8;id=10386761;file:///Users/mac/Documents/Projects/SportsBettingPoisson+ML/.venv/lib/python3.14/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=10386762;file:///Users/mac/Documents/Projects/SportsBettingPoisson+ML/.venv/lib/python3.14/site-packages/soccerdata/_config.py#91\91]8;;\
                             /Users/mac/soccerdata/config/teamname_replacements.json.                              

                    INFO     No custom league dict found. You can configure additional leagues in    ]8;id=10386768;file:///Users/mac/Documents/Projects/SportsBettingPoisson+ML/.venv/lib/python3.14/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=10386769;file:///Users/mac/Documents/Projects/SportsBettingPoisson+ML/.venv/lib/python3.14/site-packages/soccerdata/_config.py#189\189]8;;\
                             /Users/mac/soccerdata/config/league_dict.json.                                        

Season: 2024/2025
League ID: 47


In [2]:
# NBVAL_SKIP
client = SoccerDataClient(
    config=SoccerDataConfig(
        league=config["ingestion"]["soccerdata"]["league"],
        # no_cache=config["ingestion"]["soccerdata"]["no_cache"],
        # no_store=config["ingestion"]["soccerdata"]["no_store"],
        # headless=config["ingestion"]["soccerdata"]["headless"],
    )
)

try:
    fixtures = client.fetch_fixtures(season=season, league_id=league_id)
    print(f"✓ Fetched {len(fixtures)} fixtures from FBref")
except Exception as e:
    print(f"✗ Error fetching fixtures: {e}")
    fixtures = None

[08/20/26 03:54:31] INFO     Saving cached data to /Users/mac/soccerdata/data/FBref                  ]8;id=10386776;file:///Users/mac/Documents/Projects/SportsBettingPoisson+ML/.venv/lib/python3.14/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=10386777;file:///Users/mac/Documents/Projects/SportsBettingPoisson+ML/.venv/lib/python3.14/site-packages/soccerdata/_common.py#250\250]8;;\


*** Getting chromedriver 151.0.7922.138 (Previous Version)

https://storage.googleapis.com/chrome-for-testing-public/151.0.7922.138/mac-arm64/chromedriver-mac-arm64.zip ...
Download Complete!

Extracting ['chromedriver'] from chromedriver-mac-arm64.zip:
Unzip Complete!

['chromedriver'] was saved to:
/Users/mac/Documents/Projects/SportsBettingPoisson+ML/.venv/lib/python3.14/site-packages/seleniumbase/drivers/
chromedriver

[chromedriver 151.0.7922.138] is ready for use!

✓ Fetched 380 fixtures from FBref


In [ ]:
# NBVAL_SKIP
# Fetch match statistics (all fixtures)
try:
    if fixtures is not None and len(fixtures) > 0:
        match_stats_list = []
        for fixture_id in fixtures["fixture_id"].unique():
            try:
                stats = client.fetch_match_stats(fixture_id=fixture_id)
                if stats is not None and len(stats) > 0:
                    stats["fixture_id"] = fixture_id
                    match_stats_list.append(stats)
            except Exception as e:
                print(f"  Warning: Could not fetch stats for fixture {fixture_id}: {e}")
        
        if match_stats_list:
            match_stats = pd.concat(match_stats_list, ignore_index=True)
            print(f"✓ Fetched stats for {match_stats['fixture_id'].nunique()} fixtures")
        else:
            match_stats = None
            print("✗ No match stats retrieved")
    else:
        match_stats = None
except Exception as e:
    print(f"✗ Error fetching match stats: {e}")
    match_stats = None

In [ ]:
# NBVAL_SKIP
# Fetch player minutes (all fixtures)
try:
    if fixtures is not None and len(fixtures) > 0:
        player_minutes_list = []
        for fixture_id in fixtures["fixture_id"].unique():
            try:
                minutes = client.fetch_player_minutes(fixture_id=fixture_id)
                if minutes is not None and len(minutes) > 0:
                    minutes["fixture_id"] = fixture_id
                    player_minutes_list.append(minutes)
            except Exception as e:
                print(f"  Warning: Could not fetch player minutes for fixture {fixture_id}: {e}")
        
        if player_minutes_list:
            player_minutes = pd.concat(player_minutes_list, ignore_index=True)
            print(f"✓ Fetched player minutes for {player_minutes['fixture_id'].nunique()} fixtures")
        else:
            player_minutes = None
            print("✗ No player minutes retrieved")
    else:
        player_minutes = None
except Exception as e:
    print(f"✗ Error fetching player minutes: {e}")
    player_minutes = None

In [ ]:
# Create output directory
output_dir = Path(f"data/raw/{season.replace('/', '-')}")
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {output_dir}")

# Save fixtures
if fixtures is not None and len(fixtures) > 0:
    fixtures_path = output_dir / "fixtures.parquet"
    fixtures.to_parquet(fixtures_path, index=False)
    print(f"✓ Saved {len(fixtures)} fixtures to {fixtures_path}")
else:
    print("✗ No fixtures to save")

# Save match statistics
if match_stats is not None and len(match_stats) > 0:
    stats_path = output_dir / "match_stats.parquet"
    match_stats.to_parquet(stats_path, index=False)
    print(f"✓ Saved {len(match_stats)} match stat rows to {stats_path}")
else:
    print("✗ No match stats to save")

# Save player minutes
if player_minutes is not None and len(player_minutes) > 0:
    minutes_path = output_dir / "player_minutes.parquet"
    player_minutes.to_parquet(minutes_path, index=False)
    print(f"✓ Saved {len(player_minutes)} player minute rows to {minutes_path}")
else:
    print("✗ No player minutes to save")

In [ ]:
# Display sample output for documentation
if fixtures is not None and len(fixtures) > 0:
    print("Sample Fixtures (first 5 rows):")
    print(fixtures.head())
    print()

if match_stats is not None and len(match_stats) > 0:
    print("Sample Match Stats (first 5 rows):")
    print(match_stats.head())
    print()

if player_minutes is not None and len(player_minutes) > 0:
    print("Sample Player Minutes (first 5 rows):")
    print(player_minutes.head())
    print()

In [ ]:
# Validate outputs
print("=== Data Validation ===")

if fixtures is not None and len(fixtures) > 0:
    print(f"✓ Fixtures: {len(fixtures)} rows, {len(fixtures.columns)} columns")
    required_fixture_cols = ["fixture_id", "home_team", "away_team", "date"]
    missing = [c for c in required_fixture_cols if c not in fixtures.columns]
    if missing:
        print(f"  ⚠ Missing columns: {missing}")
    else:
        print(f"  ✓ All required columns present")

if match_stats is not None and len(match_stats) > 0:
    print(f"✓ Match Stats: {len(match_stats)} rows, {len(match_stats.columns)} columns")
    print(f"  - {match_stats['fixture_id'].nunique()} unique fixtures")

if player_minutes is not None and len(player_minutes) > 0:
    print(f"✓ Player Minutes: {len(player_minutes)} rows, {len(player_minutes.columns)} columns")
    print(f"  - {player_minutes['fixture_id'].nunique()} unique fixtures")
    print(f"  - {player_minutes['player_id'].nunique()} unique players")

print("\n=== Ingestion Complete ===")